In [1]:
import os
import pandas as pd
import numpy as np
import pickle
import boto3
from tqdm import tqdm

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'

Project: 20231010-gen-xii


### Make output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Write ```functions.py```

In [4]:
%%writefile functions.py

import pandas as pd
from io import StringIO
import numpy as np
import datetime as dt
import xml.etree.ElementTree as ET

# helper for get data
def get_data_helper(dict_json_request):
    # get rows
    list_dict_data = dict_json_request['rows']
    # get id and tables
    list_unique_id = []
    dict_list_tables = {}
    for dict_data in list_dict_data:
        # get unique_id
        unique_id = dict_data['row_id']
        list_unique_id.append(unique_id)
        # assign tables
        dict_list_tables[unique_id] = dict_data['sources']
    # return
    return list_unique_id, dict_list_tables

# get application table
def get_application_table(str_values, unique_id, dict_list_errors):
    # convert str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if X is empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Application Table')
        # create a single row df filled with NaN
        list_cols = [
            'app_was_empty',
        ]
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
        # set date to today
        X['applicationdate'] = dt.datetime.today()
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
    # add suffix to end of column names
    X.columns = [f'{col}__app' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# get income table
def get_income_table(str_values, unique_id, dict_list_errors):
    # list of columns for income
    list_cols = [
        'bitinvalid',
        'bituse',
        'fltgrossmonthly',
    ]
    # list of cols after aggregation
    list_cols_agg = [
        'fltgrossmonthly__income_sum',
        'fltgrossmonthly__income_count',
    ]
    # convert str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if X is empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Income Table')
        X = pd.DataFrame({col: np.nan for col in list_cols_agg}, index=[0])
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
        # rm T/F
        X_tmp = X * 1
        # filter rows
        X_tmp = X_tmp[X_tmp['bitinvalid']==0] # False
        X_tmp = X_tmp[X_tmp['bituse']==1] # True
        # aggregate
        X = pd.DataFrame()
        X['fltgrossmonthly__income_sum'] = [np.sum(X_tmp['fltgrossmonthly'])]
        X['fltgrossmonthly__income_count'] = [X_tmp.shape[0]]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# get ln table
def get_lexis_nexis_table(str_values, unique_id, dict_list_errors):
    # read str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Lexis Nexis Table')
        # create row with NaN
        list_cols = [
            'ln_was_empty',
        ]
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
    # add suffix to end of column names
    X.columns = [f'{col}__ln' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# define helper to convert to proper dtype
def helper_convert_dtype(str_col_value):
    # try converting to integer
    try:
        str_col_value = int(str_col_value)
    except:
        # try converting to float
        try:
            str_col_value = float(str_col_value)
        # leave as string
        except:
            pass
    # return
    return str_col_value

# define helper to parse tuxml
def helper_parse_tuxml(str_values):
    # get root
    root = ET.fromstring(str_values)
    # empty dict
    dict_tuxml = {}
    # iterate through child branches
    for child in root.iter(tag='{http://www.transunion.com/namespace}characteristic'):
        # get col name
        str_col_name = child.find('{http://www.transunion.com/namespace}id').text.lower()
        # get col val
        try:
            str_col_value = helper_convert_dtype(str_col_value=child.find('{http://www.transunion.com/namespace}value').text)
        # if its nonetype
        except AttributeError:
            str_col_value = np.nan
        # assign
        dict_tuxml[str_col_name] = str_col_value
    # convert dtype and return df
    return pd.DataFrame(dict_tuxml, index=[0])

# get tu table
def get_transunion_table(str_values, unique_id, dict_list_errors):
    # parse xml
    X = helper_parse_tuxml(
        str_values=str_values, 
    )
    # if empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing TU Table')
        list_cols = [
            'tu_was_empty',
        ]
        # create row with NaN
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
    else:
        pass
    # add suffix to end of column names
    X.columns = [f'{col}__tu' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

Overwriting functions.py


### Write ```api.py```

In [5]:
%%writefile api.py

from functions import *
import time

# define class
class ParsePayload:
    # init
    def __init__(self, list_cols_raw_all):
        self.str_datecol = 'applicationdate'
        self.dict_output = {}
        self.list_cols_raw_all = list_cols_raw_all
    # get data
    def get_data(self, dict_json_request):
        # start time
        time_start = time.perf_counter()

        # use helper
        list_unique_id, dict_list_tables = get_data_helper(
            dict_json_request=dict_json_request,
        )

        # save to object now for uniformity later
        self.dict_output['list_unique_id'] = list_unique_id

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print
        print(f'{self.dict_output["list_unique_id"]}: Get Data: {flt_sec:0.5f} sec.')

        # save to object
        self.dict_json_request = dict_json_request
        # save to dict_output
        self.dict_output['flt_sec_get_data'] = flt_sec
        self.dict_output['dict_list_tables'] = dict_list_tables
        # return object
        return self
    # parse data
    def parse_data(self):
        # start time
        time_start = time.perf_counter()

        # empty dict
        dict_list_x = {}
        # another empty dict
        dict_list_errors = {}
        # iterate through unique ids
        for unique_id in self.dict_output['list_unique_id']:
            # put empty list in dict errors
            dict_list_errors[unique_id] = []
            # empty list
            dict_list_x[unique_id] = []
            # get tables
            list_dict_tables = self.dict_output['dict_list_tables'][unique_id]
            # iterate through tables
            for dict_table in list_dict_tables:
                # get str_values
                str_values = dict_table['values']
                # Application table
                if dict_table['name'] == 'Application':
                    # get application table
                    X, dict_list_errors = get_application_table(
                        str_values=str_values, 
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # Income table
                elif dict_table['name'] == 'Incomes':
                    # get income table
                    X, dict_list_errors = get_income_table(
                        str_values=str_values, 
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # LexisNexis
                elif dict_table['name'] == 'Lexis Nexis Risk View 5':
                    # get ln table
                    X, dict_list_errors = get_lexis_nexis_table(
                        str_values=str_values,
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # TU
                elif dict_table['name'] == 'TUXML':
                    # get tu table
                    X, dict_list_errors = get_transunion_table(
                        str_values=str_values,
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                else:
                    pass

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print
        print(f'{self.dict_output["list_unique_id"]}: Parse Data: {flt_sec:0.5f} sec.')

        # save to dict_output
        self.dict_output['flt_sec_parse_data'] = flt_sec
        self.dict_output['dict_list_x'] = dict_list_x
        self.dict_output['dict_list_errors'] = dict_list_errors
        # return object
        return self
    # create X
    def create_x(self):
        # start time
        time_start = time.perf_counter()

        # empty list
        list_x_cbind = []
        # concatenate cols of df in each list
        for list_x in self.dict_output['dict_list_x'].values():
            # concatenate
            x_cbind = pd.concat(list_x, axis=1)
            # append
            list_x_cbind.append(x_cbind)
        # concatenate rows
        X = pd.concat(list_x_cbind, axis=0)
        
        # ensure there is a field for every feature
        for col in self.list_cols_raw_all:
            if col not in list(X.columns):
                X[col] = np.nan

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print()
        print(f'{self.dict_output["list_unique_id"]}: Create X: {flt_sec:0.5f} sec.')

        # save to dict_output
        self.dict_output['flt_sec_create_x'] = flt_sec
        self.dict_output['X_raw'] = X
        # return object
        return self

Overwriting api.py


### Get all the features from raw data

In [6]:
%%time

str_filename = 'df_train_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
list_cols_raw_all = list(pd.read_parquet(str_uri).columns)
# rm base
list_cols_raw_all = [col for col in list_cols_raw_all if '__base' not in col]
# income
list_cols_ignore = [
    'fltgrossmonthly__income_min',
    'fltgrossmonthly__income_max',
    'fltgrossmonthly__income_median',
    'fltgrossmonthly__income_mean',
    'fltgrossmonthly__income_std',
]
list_cols_raw_all = [col for col in list_cols_raw_all if col not in list_cols_ignore]
print(f'There are {len(list_cols_raw_all)} total raw features')

There are 2464 total raw features
CPU times: user 5.19 s, sys: 2.07 s, total: 7.26 s
Wall time: 2.19 s


### Initialize class

In [7]:
from api import ParsePayload

cls_parser = ParsePayload(
    list_cols_raw_all=list_cols_raw_all,
)

### Save

In [8]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))